In [10]:
import os
import numpy as np
import pandas as pd
import h3

os.environ['HAVEN_DATABASE'] = 'haven'
os.environ['AWS_PROFILE'] = 'admin'

from mirrorverse.utils import read_data_w_cache
from haven.db import write_data

In [11]:
sql = '''
select 
    h3_index,
    neighbor
from 
    options_w_angles
'''
data = read_data_w_cache(sql)
print(data.shape)
data.head()

(267734, 2)


,h3_index,neighbor
0,84222e3ffffffff,84222e3ffffffff
1,84222e3ffffffff,84222edffffffff
2,84222e3ffffffff,84222adffffffff
3,84222e3ffffffff,84222e9ffffffff
4,84222e3ffffffff,84222e1ffffffff


In [12]:
data['lat'] = data['neighbor'].apply(lambda r: h3.h3_to_geo(r)[0])
data['lon'] = data['neighbor'].apply(lambda r: h3.h3_to_geo(r)[1])
data['origin_lat'] = data['h3_index'].apply(lambda r: h3.h3_to_geo(r)[0])
data['origin_lon'] = data['h3_index'].apply(lambda r: h3.h3_to_geo(r)[1])
data.head()

,h3_index,neighbor,lat,lon,origin_lat,origin_lon
0,84222e3ffffffff,84222e3ffffffff,44.563724,-169.928944,44.563724,-169.928944
1,84222e3ffffffff,84222edffffffff,44.074344,-168.988330,44.563724,-169.928944
2,84222e3ffffffff,84222adffffffff,45.151533,-169.387859,44.563724,-169.928944
3,84222e3ffffffff,84222e9ffffffff,43.902334,-169.489574,44.563724,-169.928944
4,84222e3ffffffff,84222e1ffffffff,44.319687,-169.456075,44.563724,-169.928944


In [13]:
data['movement_heading'] = data.apply(
    lambda r: np.arctan2(
        r['lat'] - r['origin_lat'], r['lon'] - r['origin_lon']
    ),
    axis=1
)
data.head()

,h3_index,neighbor,lat,lon,origin_lat,origin_lon,movement_heading
0,84222e3ffffffff,84222e3ffffffff,44.563724,-169.928944,44.563724,-169.928944,0.000000
1,84222e3ffffffff,84222edffffffff,44.074344,-168.988330,44.563724,-169.928944,-0.479737
2,84222e3ffffffff,84222adffffffff,45.151533,-169.387859,44.563724,-169.928944,0.826763
3,84222e3ffffffff,84222e9ffffffff,43.902334,-169.489574,44.563724,-169.928944,-0.984424
4,84222e3ffffffff,84222e1ffffffff,44.319687,-169.456075,44.563724,-169.928944,-0.476428


In [14]:
data = data[['h3_index', 'neighbor', 'movement_heading']]
data.head()

,h3_index,neighbor,movement_heading
0,84222e3ffffffff,84222e3ffffffff,0.000000
1,84222e3ffffffff,84222edffffffff,-0.479737
2,84222e3ffffffff,84222adffffffff,0.826763
3,84222e3ffffffff,84222e9ffffffff,-0.984424
4,84222e3ffffffff,84222e1ffffffff,-0.476428


In [15]:
data['version'] = 1
write_data(data, 'movement_headings', ['version'])

In [16]:
from haven.db import drop_table

drop_table('movement_model_full_features_10')